# Day 4: Baseline RAG with citations

Study guide: `course/week-1/day-04-baseline-rag/README.md`

Goal: question in → 5 nearest chunks → prompt → Llama → answer with citations, or "I don't know".

**Setup:** reload edited `.py` files automatically. Run once after every kernel restart.

In [2]:
%load_ext autoreload
%autoreload 2

## Step 1: Search as project code
`search(question, k)` embeds the question with MiniLM and runs `ORDER BY embedding <=> question LIMIT k` in pgvector. It returns `RetrievedChunk` objects with `similarity = 1 - distance`.

**Findings:**
- "Does LoRA add inference latency?" → top hit `2106.09685.pdf::0021` (p.4) at **0.648**; the answer chunk ::0024 (p.5) is 4th at 0.564.
- 2 of the top 5 come from the **QLoRA** paper (it also talks about LoRA), so the top 5 mixes papers.
- The "Loading weights" / HF_TOKEN lines are just MiniLM loading on the first call; harmless.

In [3]:
from research_rag.retrieval.search import search

results = search("Does LoRA add inference latency?", k=5)

for result in results:
    print(f"{result.similarity:.3f}  {result.id:<24} p.{result.page_number:<3} {result.text[:80]}...")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 10697.90it/s]


0.648  2106.09685.pdf::0021     p.4   LoRA roughly converges to training the original model, while adapter-based metho...
0.586  2305.14314.pdf::0030     p.6   under what circumstances slowdowns occur from the paging process. Default LoRA h...
0.584  2106.09685.pdf::0020     p.4   a result, we simply set α to the first r we try and do not tune it. This scaling...
0.564  2106.09685.pdf::0024     p.5   350GB to 35MB)4. This allows us to train with significantly fewer GPUs and avoid...
0.546  2305.14314.pdf::0124     p.22  65B). We find LoRA r is unrelated to final performance if LoRA is used on all la...


## Step 2: The prompt
`build_prompt()` = rules + numbered context `[n] (source, page p)` + question. Prompt **v1** (`PROMPT_VERSION`).

**Findings:**
- Each chunk gets a number, so the model can cite `[1]`, and we can map `[1]` back to a source and page.
- The rules come **first**: use ONLY the context, cite every fact, and if the answer isn't there, reply exactly with `NO_ANSWER`.
- Prompts are versioned like code: when we change the prompt (Day 7), we can compare v1 and v2 on the golden set.

In [6]:
from research_rag.generation.prompts import PROMPT_VERSION, build_prompt

question = "Does LoRA add inference latency?"
results = search(question, k=5)
prompt = build_prompt(question, results)

print("Prompt version:", PROMPT_VERSION)
print(prompt)

Prompt version: v1
You are a careful research assistant.
Answer the question using ONLY the numbered context below.

Rules:
- Cite every fact with the number of the context it came from, like [1] or [2][3].
- If the context does not contain the answer, reply exactly: "I don't know based on the provided papers."
- Do not use outside knowledge. Keep the answer short.

Context:
[1] (2106.09685.pdf, page 4)
LoRA roughly converges to training the original model, while adapter-based methods converges to an MLP and prefix-based methods to a model that cannot take long input sequences. No Additional Inference Latency. When deployed in production, we can explicitly compute and store W = W0 + BA and perform inference as usual. Note that both W0 and BA are in Rd×k. When we need to switch to another downstream task, we can recover W0 by subtracting BA and then adding a different B′A′, a quick operation with very little memory overhead. Critically, this 2They represent a negligible number of parame

## Step 3: The LLM client
`generate(prompt)` posts to Ollama `/api/generate` with **temperature 0** and **`num_ctx` 8192**, retries once, and raises `LLMError` on failure.

**Findings:**
- The answer was correct and grounded: LoRA merges `W = W0 + BA` before deployment, so **no extra inference latency**, citing [1] and [4].
- Temperature 0 = the same answer every time (less randomness), **not** more accuracy.
- `num_ctx` matters: Ollama's default context is smaller, and 5 chunks plus the rules could get silently cut off.

In [7]:
from research_rag.generation.llm import generate

answer = generate(prompt)    # the prompt you built in Step 2
print(answer)

No. According to [1] and [4], when deployed in production, we can explicitly compute and store W = W0 + BA and perform inference as usual, and when we need to switch to another downstream task, we can recover W0 by subtracting BA and then adding a different B′A′, a quick operation with very little memory overhead.


## Step 4: Citations
`extract_citations()` finds `[n]` in the answer with a regex and maps it to `chunks[n-1]`. It drops repeats and numbers that don't exist (e.g. `[9]` with only 5 chunks).

**Findings:**
- `[1]` → `2106.09685.pdf` p.4 and `[4]` → p.5: both real LoRA pages, so the citations are checkable.
- The model cited 2 of the 5 chunks: it ignored the QLoRA chunks, which is good.

In [8]:
from research_rag.generation.citations import extract_citations

citations = extract_citations(answer, results)
for citation in citations:
    print(f"[{citation.number}] {citation.source}, page {citation.page_number}  ({citation.chunk_id})")

[1] 2106.09685.pdf, page 4  (2106.09685.pdf::0021)
[4] 2106.09685.pdf, page 5  (2106.09685.pdf::0024)


## Step 5: answer()
`rag.answer()` runs the whole pipeline in one call: search → prompt → generate → citations, and returns an `Answer` with `answered` (False if the reply is `NO_ANSWER`), `prompt_version` and `latency_ms`. `make ask Q="..."` calls the same function from the terminal.

In [ ]:
from research_rag.rag import answer

result = answer("Does LoRA add inference latency?")

print(result.text)
print()
print("answered:", result.answered, "| prompt:", result.prompt_version, "| latency:", result.latency_ms, "ms")
for citation in result.citations:
    print(f"  [{citation.number}] {citation.source}, page {citation.page_number}")

## Step 6: Try real questions
Ran 9 questions with `make ask` (prompt v1, k=5):

| Question | Result | Cause |
| --- | --- | --- |
| Does LoRA add latency? | ✅ correct, cites p.4–5 | – |
| Banana bread | ✅ "I don't know" | – |
| World Cup | ✅ "I don't know" | – |
| QLoRA memory | ⚠️ partial (paging only, missed NF4) | answer spread over several chunks |
| HyDE | ⚠️ vague | weak chunks |
| NF4 | ⚠️ appendix numbers from p.24 | acronym; weak retrieval |
| Lost in the middle | ❌ false refusal | **retrieval** |
| ReAct | ❌ false refusal | **generation + chunking** |

Latency: 3.6–9 s per answer. The cell below checks the two false refusals: **was the answer in the retrieved chunks?**

In [9]:
for question in ["What is the lost in the middle problem?", "What does ReAct combine?"]:
    print("Q:", question)
    for result in search(question, k=5):
        print(f"  {result.similarity:.3f}  {result.id:<24} p.{result.page_number:<3} {result.text[:90]}...")
    print()

Q: What is the lost in the middle problem?
  0.465  2201.11903.pdf::0222     p.43  58 - 23 = 35 balls. On Wednesday he lost 2 more so now he has 35 - 2 = 33 balls. The answe...
  0.446  2201.11903.pdf::0217     p.42  Michael started with 58 golf balls and lost 23, so he has 58 - 23 = 35. After he lost 2 mo...
  0.436  2201.11903.pdf::0187     p.35  tuesday, he had 58 - 23 = 35. After losing 2 more, he had 35 - 2 = 33 golf balls. The answ...
  0.428  2201.11903.pdf::0214     p.42  and there are 15 trees in the beginning, so the workers plant 21 - 15 = 6 trees. The answe...
  0.426  2201.11903.pdf::0184     p.35  . Then there were 21 trees after some more were planted. So there must have been 21 - 15 =...

Q: What does ReAct combine?
  0.462  2210.03629.pdf::0053     p.8   Act is the first demonstration of combined reasoning and action using an LLM applied to an...
  0.430  2210.03629.pdf::0039     p.6   flexibility in formulating reasoning steps, leading to more reasoning error rate tha

**Diagnosis:**
- **"Lost in the middle" → retrieval failure.** All 5 chunks are Chain-of-Thought golf-ball maths ("he **lost** 2 more balls…", 0.43–0.47). MiniLM matched the word "lost", not the paper's name. The right paper isn't in the top 5, so **refusing was correct**. Fix later: keyword/hybrid search (Day 8), reranking (Day 9).
- **"ReAct" → generation + chunking miss.** Top chunk `2210.03629.pdf::0053` says "…the first demonstration of **combined reasoning and action**", which is the answer. But the chunk starts mid-word with **"Act is…"** (the "Re" is at the end of the previous chunk), so the model couldn't be sure it was about ReAct and refused. Fix later: sentence-aware chunks and prompt tuning (Day 7).

**Rule:** when an answer is wrong, first check whether the answer was in the retrieved chunks. That splits "wrong answer" into a **retrieval** problem or a **generation** problem.

## Step 7: Tests
`tests/test_prompts.py` (2), `tests/test_citations.py` (3), `tests/test_rag.py` (2) → **28 tests** in total.

`test_rag.py` uses **mocking**: `monkeypatch.setattr(rag, "search", fake_search)` and `(rag, "generate", fake_generate)` replace the real database and Llama with fakes for one test. So we test our own logic (citations, the "I don't know" flag) in under a second, without Postgres or Ollama running.

## Findings
- **Built:** a working baseline RAG with prompt v1, k=5, Llama 3.1 8B, citations, `make ask`, and 7 mocked tests (28 total).
- **Baseline by eye:** 1 fully correct · 2 correct refusals · 3 partial/vague · 2 false refusals, out of 9 questions (the Step 6 table).
- **Two kinds of failure:** retrieval (the right chunk isn't found, as with "lost in the middle") vs generation/chunking (found, but not used, as with "ReAct").
- **Embeddings are weak on names and exact phrases** ("lost in the middle", "NF4"); keyword search should help (Day 8).
- **Next:** stop judging by eye. Days 5–6 build a golden dataset and score retrieval with numbers.